In [76]:
#
# microPAM Viewer (WMXZ) 24-03-2026
# inspired by Nauta scientific (M.M.)
#=============================================================================
import os
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import (FigureCanvasTkAgg, NavigationToolbar2Tk)

import scipy.signal as signal

import tkinter as tk
from tkinter import ttk, filedialog, messagebox

import threading

import microPAM as pam

In [77]:
def load_wav(filepath):
    return pam.load_microPAM(filepath)

def compute_psd_matrix(audio, rate,
                        window_sec=1.0, overlap_frac=0.5,
                        nperseg_factor=0.05,
                        nfft=1024,
                        freq_min=20000, freq_max=None,
                        progress_cb=None):
    if freq_max is None:
        freq_max = rate / 2
    freq_max     = min(freq_max,rate/2)
    #
    win_samples  = int(window_sec * rate)
    step_samples = max(1, int(win_samples * (1 - overlap_frac)))
    nperseg      = min(int(rate * nperseg_factor), win_samples, nfft//2)
    n_samples    = len(audio)
    starts       = np.arange(0, n_samples, step_samples)
    times        = (starts + win_samples / 2) / rate
    n_steps      = len(starts)

    psd_list = []
    for k, s in enumerate(starts):
        s1=s
        s2=min(s+win_samples,len(audio))
        chunk = audio[s1:s2]
        f, pxx = signal.welch(chunk, fs=rate, nperseg=nperseg, nfft=nfft,
                              window='hann', scaling='density')
        psd_list.append(pxx)
        if progress_cb and k % max(1, n_steps // 50) == 0:
            progress_cb(k / n_steps)

    freqs   = f
    psd_mat = np.array(psd_list)
    mask    = (freqs >= freq_min) & (freqs <= freq_max)
    freqs   = freqs[mask]
    psd_mat = psd_mat[:, mask]
    psd_db  = 10 * np.log10(np.maximum(psd_mat, 1e-20))
    return times, freqs, psd_db


def compute_impulsivity_index(psd_db, percentile_high=95):
    median_f = np.median(psd_db, axis=0)
    p_high_f = np.percentile(psd_db, percentile_high, axis=0)
    diff_map = psd_db - median_f[np.newaxis, :]
    ii       = np.mean(np.maximum(diff_map, 0), axis=1)
    return ii, diff_map, median_f, p_high_f


def detect_events(times, ii, threshold_factor=2.0, min_gap_sec=0.05):
    mean_ii   = np.mean(ii)
    std_ii    = np.std(ii)
    threshold = mean_ii + threshold_factor * std_ii
    dt        = (times[1] - times[0]) if len(times) > 1 else 1.0
    min_gap_n = max(1, int(min_gap_sec / dt))
    above     = ii > threshold
    events    = []
    last_idx  = -min_gap_n - 1
    for i, (t, val, flag) in enumerate(zip(times, ii, above)):
        if flag and (i - last_idx) > min_gap_n:
            events.append({'time': t, 'ii_value': val, 'index': i})
            last_idx = i
    return events, threshold


def analyze_file(filepath, params, perc, thr_factor, min_gap):
    """Analizza un singolo file. Restituisce un dict con tutti i risultati."""
    rate, audio = load_wav(filepath)
    #
    fmin=params['freq_min']
    times, freqs, psd_db = compute_psd_matrix(audio, rate,freq_min=fmin)
    #
    ii, diff_map, median_f, p_high_f = compute_impulsivity_index(psd_db, perc)
    #
    events, threshold = detect_events(times, ii, thr_factor, min_gap)
    return {
        "filepath": filepath,
        "rate":     rate,
        "duration": len(audio) / rate,
        "times":    times,
        "freqs":    freqs,
        "psd_db":   psd_db,
        "mean_db":  10*np.log10(np.mean(10**(psd_db/10),axis=0)),
        "ii":       ii,
        "diff_map": diff_map,
        "median_f": median_f,
        "p_high_f": p_high_f,
        "events":   events,
        "threshold": threshold,
        "n_events": len(events),
        "mean_ii":  float(np.mean(ii)),
        "max_ii":   float(np.max(ii)),
    }


In [78]:
class paramClass(tk.Toplevel):
    def __init__(self, master, txtvar,titles, groups, **kwargs):
        super().__init__(master, **kwargs)

        cols=np.array([np.double(txtvar[1][ii].get()).astype(int) for ii in range(len(txtvar[0]))])
        #
        frame1=tk.Frame(self,border=1,borderwidth=1,relief="solid",padx=5,pady=5)
        frame1.pack()
        #
        for kk in range(max(groups)[0]+1):
            framex=tk.Frame(frame1,bd = 1,relief='solid',padx=5,pady=5)
            framex.grid(row=0,column=kk,sticky='N')
            #
            for jj in range(max(cols)+1):
                frame2=tk.LabelFrame(framex,text=titles[jj],padx=5,pady=5)
                colx=np.where((cols==jj )& (groups[jj][0]==kk))[0]
                #
                for ii,col in enumerate(colx):
                    itype=np.double(txtvar[3][col].get()).astype(int)
                    if itype==0:
                        w=10
                    else:
                        w=itype
                    lbl1 = tk.Label(frame2, text=txtvar[0][col])
                    ent1 = tk.Entry(frame2, textvariable=txtvar[2][col],width=w) 
                    lbl2 = tk.Label(frame2, text=txtvar[4][col])
                    #
                    lbl1.grid(row=ii,column=0,padx=10,pady=5,sticky="E")
                    ent1.grid(row=ii,column=1,padx=10,pady=5,sticky="W")
                    lbl2.grid(row=ii,column=2,padx=10,pady=5,sticky="W")
                column,row=groups[jj]
                frame2.grid(row=row,column=column,sticky='N')

        self.wait_window()
        return


In [79]:
#=================================================================
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.wm_title("microPAM Viewer (WMXZ)")
        self.geometry("1400x800+10+10")
        #
        self.file_list   = []   # list of file paths
        self.results     = {}   # path → result dict
        self.file_status = {}   # path → status string
        
        self.selected_file = None

        self._stop_requested=False
        self._running = -1
        #
        # default parameters
        self.param_titles=['General','Time window', 'Spectral Window', 'PSD', 'Detector']
        self.param_groups=[[0,0],     [1,0],        [1,1],             [1,2],  [2,0]]
        self.param={
                    'ich':   [0, 0,    2,'channel #'],
                    'diff':  [0, 0,    2,'diff filter'],
                    'nfft':  [0, 1024, 0,'FFT size [pts]'],
                    'iplt':  [0, 0,    2,'do plotting'],
                    'twin':  [1, 1,    0,'time window [s]'],
                    'over':  [1, 0.5,  0,'Overlap [rel]'],
                    'fmin':  [2, 1000, 0,'Freq. min [Hz]'],
                    'fmax':  [2, 48000,0,'Freq. max [Hz]'],
                    'frac':  [3, 0.05, 0,'rel. spect. window'],
                    'perc':  [4, 95,   0,'signal percentile'],
                    'thres': [4, 2,    0,'threshold'],
                    'gap':   [4, 0.05, 0,'min det. gap']}
        #
        self._build_ui()

    def _build_ui(self):
        # ── Menu ──
        menu = tk.Menu()
        self.config(menu=menu)
        self._build_menu(menu)
        
        # ── Top toolbar ──
        toolbar = tk.Frame(self, pady=7, padx=10,bd = 1,relief='solid')
        toolbar.pack(fill="x", side="top")
        self._build_toolbar(toolbar)

        # ── main area left: file list ── right: plotting area -──
        body = tk.Frame(self,bd = 0,relief='solid')
        body.pack(fill="both", expand=True)

        ww=350
        left = tk.Frame(body, width=ww, padx=8, pady=8,bd = 1,relief='solid')
        left.pack(fill="y", side="left")
        left.pack_propagate(False)

        # Summary
        sum = tk.LabelFrame(left, width=ww, height=100, padx=1, pady=1,bd = 1,relief='solid',text='Summary')
        sum.pack(fill="y", side="top")
        self._build_summary(sum)

        # File list
        flist = tk.LabelFrame(left, width=ww, bd = 1,relief='solid',text='File list')
        flist.pack(fill="y", expand=True, side='bottom')
        self._build_file_list(flist)

        # plot area
        right = tk.Frame(body,bd = 0,relief='solid', padx=8)
        right.pack(fill="both", expand=True, side="left")
        self._build_plots(right)

    #----------------------------------------------------------------------------------
    def _build_menu(self,menu):
        fileMenu = tk.Menu(menu,tearoff=0)
        fileMenu.add_command(label="Edit",command=self._edit_parameters)
        fileMenu.add_command(label="Load",command=self._load_parameters)
        fileMenu.add_command(label="Save",command=self._save_parameters)

        menu.add_cascade(label="Parameters", menu=fileMenu)
    
    def _edit_parameters(self):
        #encode for passing to paramClass
        param_keys=list(self.param.keys())
        txtvar = [param_keys,                                                   # 0
                  [tk.StringVar(value=self.param[x][0]) for x in param_keys],   # 1
                  [tk.StringVar(value=self.param[x][1]) for x in param_keys],   # 2
                  [tk.StringVar(value=self.param[x][2]) for x in param_keys],   # 3
                  [                   self.param[x][3]  for x in param_keys]]   # 4    
        #get input
        paramClass(self, txtvar,self.param_titles,self.param_groups)
        #decode
        for ii,key in enumerate(param_keys):
            self.param[key][1]=np.double(txtvar[2][ii].get())
        return
    
    def _save_parameters(self):
        np.save('Parameters.npy', self.param) 
        return
    
    def _load_parameters(self):
        self.param={np.load('Parameters.npy',allow_pickle='TRUE').item()}
        return

    def _get_params(self):
        return (
            {   "window_sec":     float(self.param['twin'][1]),
                "overlap_frac":   float(self.param['over'][1]),
                "nperseg_factor": float(self.param['frac'][1]),
                "freq_min":       float(self.param['fmin'][1]),
                "freq_max":       float(self.param['fmax'][1]),
            },
            float(self.param['perc'][1]),
            float(self.param['thres'][1]),
            float(self.param['gap'][1])
        )
    
    #------------------------------------------------------------------------
    def _build_toolbar(self,parent):
        def btn(text, cmd, bold=False):
            f = ("Helvetica", 10, "bold") if bold else ("Helvetica", 10)
            b = tk.Button(parent, text=text, command=cmd,
                          font=f,  padx=11, pady=5,
                          cursor="hand2")
            b.pack(side="left", padx=3)
            return b

        btn("📁  Add Folder", self._add_folder)

        self.btn_run = btn("▶  Start", self._start_batch, bold=True)
    
    #
    def _add_folder(self):
        folder = filedialog.askdirectory()
        if not folder:
            return
        paths = []
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                if f.lower().endswith(".wav"):
                    paths.append(os.path.join(root, f))
        self._add_paths(paths)

    def _add_paths(self, paths):
        added = 0
        for p in paths:
            if p not in self.file_list:
                self.file_list.append(p)
                self.file_status[p] = 'pending'
                self.tree.insert("", "end", iid=p,
                                 values=(os.path.basename(p),'pending', "—"))
                added += 1
        self._update_summary()
        #self.status_var.set(f"Aggiunti {added} file. Totale: {len(self.file_list)}")

        self._running=0
        self._batch_done=0
        return
        #

    def _start_batch(self):
        if self._running==0:
            self.btn_run.config(text="⏹  Stop",fg='Red')
            self._running=1
            threading.Thread(target=self._batch_thread, daemon=True).start()
        else:
            self.btn_run.config(text="▶  Start",fg='Green')
            self._running=0
        return
    
    def _batch_thread(self):
        #
        self.batch_done=0
        for path in self.file_list:
            if self.file_status[path] == 'done':
                continue
            if self._running==0:
                break
            self.selected_file=path
            ret = self._process_file(path)
        self.batch_done=1
        self.after(0, self._file_batch_done())

    #-------------------------------------------------------------------------------
    def _build_summary(self,parent):
        self.lbl_summary = {}
        for key, lbl in [("total",     "Files in list"),
                         ("done",      "Files completed"),
                         ("tot_events","Tot. events")]:
            f = tk.Frame(parent)
            f.pack(fill="x", pady=1)
            #
            tk.Label(f, text=lbl,
                     font=("Helvetica", 8), width=15, anchor="e").pack(side="left")
            #
            v = tk.Label(f, text="—",
                         font=("Helvetica", 8, "bold"), anchor="e")
            v.pack(side="right")
            self.lbl_summary[key] = v
        return
    
    def _update_summary(self):
        total = len(self.file_list)
        done=0
        done  = sum(1 for s in self.file_status.values() if s == 'done')
        tot_ev=0
        tot_ev = sum(r["n_events"] for r in self.results.values())
        #
        self.lbl_summary["total"].config(text=str(total))
        self.lbl_summary["done"].config(text=str(done))
        self.lbl_summary["tot_events"].config(text=str(tot_ev))
        txt=""
    #-----------------------------------------------------------------------------
    def _build_file_list(self,parent):
        # Treeview
        list_dict={   "name": ["File",  'e',    170],
                    "status": ["Status",'center',70],
                    "events": ['Events','center',50]}

        cols= list_dict.keys()
        self.tree = ttk.Treeview(parent, columns=list(cols), show="headings",
                                  selectmode="extended")
        for key in cols:
            self.tree.heading(key, text=list_dict[key][0])            
            self.tree.column(key,  width=list_dict[key][2], anchor=list_dict[key][1])

        style = ttk.Style()
        style.theme_use("default")
        style.configure("Treeview",rowheight=22, font=("Helvetica", 8))
        style.configure("Treeview.Heading",
                         font=("Helvetica", 8, "bold"), relief="flat")
        style.map("Treeview", background=[("selected", 'lightgreen')],
                              foreground=[("selected", "black")])

        sb = ttk.Scrollbar(parent, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=sb.set)
        self.tree.pack(side="left", fill="both", expand=True)
        sb.pack(side="right", fill="y")

        self.tree.bind("<<TreeviewSelect>>", self._on_tree_select)

    def _on_tree_select(self, _event=None):
        sel = self.tree.selection()
        if not sel:
            return
        path = sel[0]

        self.selected_file = path
        if path not in self.results:
            ret=self._process_file(path)

        if self.param['iplt'][1]==0:
            self._show_result(path)

        self.tree.selection_remove(sel)
        return
    
    #-------------------------------------------------------------------------------
    def _process_file(self,path):
        self._set_file_status(path, 'running')
        self.canvas.flush_events()

        try:
            params, perc, thr, mingap = self._get_params()
        except ValueError as e:
            self.after(0, lambda: messagebox.showerror("Parameters not valid", str(e)))
            self._running = 0
            return 0

        try:
            result = analyze_file(path, params, perc, thr, mingap)
            self.results[path] = result
            self.after(0, lambda p=path, r=result: self._file_done(p, r))
        except Exception as e:
            err = str(e)
            self.after(0, lambda p=path, e=err: self._file_error(p, e))
            return 0
        return 1
    
    def _set_file_status(self, path, status):
        self.file_status[path] = status
        vals = self.tree.item(path, "values")
        self.tree.item(path, values=(vals[0], status, vals[2]))

    def _file_done(self, path, result):
        self.file_status[path] = 'done'
        self.tree.item(path, values=(
                os.path.basename(path),
                'done', 
                str(result["n_events"]) 
            )
        )
        #
        self._update_summary()
        if self.param['iplt'][1]==1:
            self._show_result(path)
        
    def _file_error(self, path, err):
        self.file_status[path] = 'error'
        print(err)
        self.tree.item(path, values=( os.path.basename(path), 'error', "—"))

    def _file_batch_done(self):
        self._show_result()

    #-------------------------------------------------------------------------------
    def _build_plots(self,parent):
        self.fig = plt.Figure(figsize=(10, 6.5),layout='constrained')#, tight_layout=True)

        # containing the Matplotlib figure
        self.canvas = FigureCanvasTkAgg(self.fig)
        
        gs = self.fig.add_gridspec(3, 1)
        self.ax_heatmap = self.fig.add_subplot(gs[0])
        self.ax_ii      = self.fig.add_subplot(gs[1])
        self.ax_psd     = self.fig.add_subplot(gs[2])

        self.ax_ii.sharex(self.ax_heatmap)

        #self._clear_plots("Select a file from the list to visualize")

        self.canvas = FigureCanvasTkAgg(self.fig, master=parent)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        nav = NavigationToolbar2Tk(self.canvas, parent)
        nav.config(bg='white')
        nav.update()
        return
    
    def _show_result(self,path=None):
        if path==None:
            # show results 
            mat=[]
            t_ext=[]
            f_ext=[]
            for path in self.file_list:
                if self.file_status[path] == 'done':
                    r = self.results[path]
                    t_ext = [r["times"][0], r["times"][-1]]
                    f_ext = [r["freqs"][0] / 1000, r["freqs"][-1] / 1000]
                    mat.append(r['mean_db'])
                else:
                    continue
            M=np.array(mat)
            print(M.shape)

            ax = self.ax_heatmap
            axm=ax.images
            if len(axm)>0:
                axm[-1].colorbar.remove()
            ax.cla()
            im    = ax.imshow(M.T, aspect='auto', origin='lower',
                            extent=[*t_ext, *f_ext],
                            cmap='inferno')
            ax.set_title(f"PSD Heatmap", fontsize=9, pad=4)
            ax.set_xlabel("Time (s)", fontsize=8)
            ax.set_ylabel("Freq (kHz)", fontsize=8)
            ax.tick_params(labelsize=7)
            
            self._colorbar = self.fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
            self._colorbar.ax.tick_params(labelsize=7)
        else:
            r = self.results[path]
            fname = os.path.basename(path)
            print(fname)

            #for ax in [self.ax_heatmap, self.ax_ii, self.ax_psd]:
            #    ax.cla()

            # ── Heatmap ───────────────────────────────────────────────────────────
            ax = self.ax_heatmap
            t_ext = [r["times"][0], r["times"][-1]]
            f_ext = [r["freqs"][0] / 1000, r["freqs"][-1] / 1000]
            vmax  = np.percentile(np.abs(r["diff_map"]), 99)
            axm=ax.images
            if len(axm)>0:
                axm[-1].colorbar.remove()
            ax.cla()
            im    = ax.imshow(r["diff_map"].T, aspect='auto', origin='lower',
                            extent=[*t_ext, *f_ext],
                            cmap='inferno', vmin=-vmax / 4, vmax=vmax)
            ax.set_title(f"PSD Heatmap — {fname}", fontsize=9, pad=4)
            ax.set_xlabel("Time (s)", fontsize=8)
            ax.set_ylabel("Freq (kHz)", fontsize=8)
            ax.tick_params(labelsize=7)
            for ev in r["events"]:
                ax.axvline(ev['time'], color='red', alpha=0.65, lw=0.8)
            
            self._colorbar = self.fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
            self._colorbar.ax.tick_params(labelsize=7)
            #
            # ── Impulsivity Index ─────────────────────────────────────────────────
            ax = self.ax_ii
            ax.cla()
            ax.plot(r["times"], r["ii"], lw=0.8, alpha=0.9, label="II(t)")
            ax.axhline(r["threshold"], color='gray', lw=1.2, ls='--',
                    label=f"Threshold ({r['threshold']:.3f})")
            ev_t = [e['time'] for e in r["events"]]
            ev_v = [e['ii_value'] for e in r["events"]]
            if ev_t:
                ax.scatter(ev_t, ev_v, color='red', s=18, zorder=5,
                        label=f"{r['n_events']} events")
            ax.set_title("Impulsivity Index II(t)", fontsize=9, pad=4)
            ax.set_xlabel("Time (s)", fontsize=8)
            ax.set_ylabel("II (dB)", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.legend(fontsize=7, loc='lower right')
            ax.grid(True, alpha=0.3)

        #
        self.canvas.draw()
        return

In [80]:
#=============================================================================
def main():
    app=App()
    app.mainloop()

if __name__ == '__main__':
    main()


(9, 497)
(29, 497)
(96, 497)
(305, 497)
(405, 497)
